## Datensammlung: BFS-Hotellerie (Tabellen 201 und 101), Wetter, Geodaten.

In [5]:
import requests, json, pathlib

BASE = "https://www.pxweb.bfs.admin.ch/api/v1/de/px-x-1003020000_{t}/px-x-1003020000_{t}.px"
RAW = pathlib.Path("../data/raw/bfs"); RAW.mkdir(parents=True, exist_ok=True)

for t in ["201", "101"]:
    meta = requests.get(BASE.format(t=t), timeout=30).json()
    (RAW / f"meta_{t}.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1), encoding="utf-8")
    print("Tabelle", t, "-", meta["title"])
    for v in meta["variables"]:
        print(f"  {v['code']:<14} {len(v['values']):>4} Werte   z.B. {v['valueTexts'][:4]}")

print()
print("--- Codes ---")
for t in ["201", "101"]:
    meta = json.loads((RAW / f"meta_{t}.json").read_text(encoding="utf-8"))
    print("Tabelle", t)
    for v in meta["variables"]:
        if v["code"] in ("Indikator", "Monat", "Herkunftsland"):
            print(" ", v["code"], list(zip(v["values"], v["valueTexts"]))[:15])



Tabelle 201 - Hotellerie: Angebot und Nachfrage der geöffneten Betriebe nach Jahr, Monat, Gemeinde und Indikator
  Jahr             14 Werte   z.B. ['2013', '2014', '2015', '2016']
  Monat            13 Werte   z.B. ['Jahrestotal', 'Januar', 'Februar', 'März']
  Gemeinde        186 Werte   z.B. ['Kloten', 'Opfikon', 'Dübendorf', 'Uster']
  Indikator         8 Werte   z.B. ['Betriebe', 'Zimmer', 'Betten', 'Ankünfte']
Tabelle 101 - Hotellerie: Ankünfte und Logiernächte der geöffneten Betriebe nach Jahr, Monat, Gemeinde, Herkunftsland und Indikator
  Jahr             14 Werte   z.B. ['2013', '2014', '2015', '2016']
  Monat            13 Werte   z.B. ['Jahrestotal', 'Januar', 'Februar', 'März']
  Gemeinde        186 Werte   z.B. ['Kloten', 'Opfikon', 'Dübendorf', 'Uster']
  Herkunftsland    73 Werte   z.B. ['Herkunftsland - Total', 'Schweiz', 'Deutschland', 'Frankreich']
  Indikator         2 Werte   z.B. ['Ankünfte', 'Logiernächte']

--- Codes ---
Tabelle 201
  Monat [('YYYY', 'Jahrestota

In [6]:
meta = json.loads((RAW / "meta_201.json").read_text(encoding="utf-8"))
ind = next(v for v in meta["variables"] if v["code"] == "Indikator")
for code, text in zip(ind["values"], ind["valueTexts"]):
    print(code, text)

1 Betriebe
2 Zimmer
3 Betten
4 Ankünfte
5 Logiernächte
6 Zimmernächte
7 Zimmerauslastung in %
8 Bettenauslastung in %


## 2. Hotellerie-Daten (Tabelle 201)

In [7]:
   import requests, json, time, pathlib
   import pandas as pd

   BASE = "https://www.pxweb.bfs.admin.ch/api/v1/de/px-x-1003020000_{t}/px-x-1003020000_{t}.px"
   RAW = pathlib.Path("../data/raw/bfs")
   meta = json.loads((RAW / "meta_201.json").read_text(encoding="utf-8"))
   gemeinden = next(v for v in meta["variables"] if v["code"] == "Gemeinde")["values"]

   def fetch_201(jahr):
       query = {
           "query": [
               {"code": "Jahr",      "selection": {"filter": "item", "values": [str(jahr)]}},
               {"code": "Monat",     "selection": {"filter": "item", "values": [str(m) for m in range(1, 13)]}},
               {"code": "Gemeinde",  "selection": {"filter": "item", "values": gemeinden}},
               {"code": "Indikator", "selection": {"filter": "item", "values": ["2","3","5","6","7","8"]}},
           ],
           "response": {"format": "json"},
       }
       r = requests.post(BASE.format(t="201"), json=query, timeout=60)
       r.raise_for_status()
       return r.json()

   for jahr in range(2013, 2027):
       ziel = RAW / f"hesta_201_{jahr}.json"
       if ziel.exists():
           continue                      # schon geladen -> überspringen
       data = fetch_201(jahr)
       ziel.write_text(json.dumps(data), encoding="utf-8")
       print(jahr, len(data["data"]), "Zeilen")
       time.sleep(2)                     # Server schonen

2013 13392 Zeilen
2014 13392 Zeilen
2015 13392 Zeilen
2016 13392 Zeilen
2017 13392 Zeilen
2018 13392 Zeilen
2019 13392 Zeilen
2020 13392 Zeilen
2021 13392 Zeilen
2022 13392 Zeilen
2023 13392 Zeilen
2024 13392 Zeilen
2025 13392 Zeilen
2026 13392 Zeilen


In [8]:
   rows = []
   for f in sorted(RAW.glob("hesta_201_*.json")):
       for d in json.loads(f.read_text(encoding="utf-8"))["data"]:
           rows.append(d["key"] + d["values"])          # [Jahr, Monat, Gemeinde, Indikator, Wert]
   df = pd.DataFrame(rows, columns=["jahr", "monat", "bfs_nr", "indikator", "wert"])
   df["wert"] = pd.to_numeric(df["wert"], errors="coerce")   # "..." -> NaN
   namen = {"2":"zimmer","3":"betten","5":"logiernaechte","6":"zimmernaechte",
            "7":"zimmerauslastung","8":"bettenauslastung"}
   df["indikator"] = df["indikator"].map(namen)
   hotel = df.pivot_table(index=["bfs_nr","jahr","monat"], columns="indikator",
                          values="wert", aggfunc="first").reset_index()
   hotel[["bfs_nr","jahr","monat"]] = hotel[["bfs_nr","jahr","monat"]].astype(int)
   hotel.to_csv("../data/processed/hotellerie_monat.csv", index=False)
   print(hotel.shape); hotel.head()

(29459, 9)


indikator,bfs_nr,jahr,monat,betten,bettenauslastung,logiernaechte,zimmer,zimmerauslastung,zimmernaechte
0,1024,2013,1,310.0,18.0,1737.0,175.0,22.0,1181.0
1,1024,2013,10,310.0,29.0,2760.0,175.0,44.0,2364.0
2,1024,2013,11,310.0,24.0,2186.0,175.0,35.0,1830.0
3,1024,2013,12,310.0,20.0,1882.0,175.0,30.0,1575.0
4,1024,2013,2,310.0,19.0,1684.0,175.0,31.0,1504.0


In [9]:
   print(hotel.shape)                         # max. 186 x 12 x 14 = 31'248 Zeilen
   print(hotel.isna().sum())                  # fehlende Werte pro Spalte, Zahl notieren
   print(hotel.describe().round(1))           # zimmerauslastung muss zwischen 0 und 100 liegen
   print(sorted(hotel["jahr"].unique()))      # 2013 bis 2026

(29459, 9)
indikator
bfs_nr               0
jahr                 0
monat                0
betten               0
bettenauslastung    81
logiernaechte       81
zimmer               0
zimmerauslastung    81
zimmernaechte       81
dtype: int64
indikator   bfs_nr     jahr    monat   betten  bettenauslastung  \
count      29459.0  29459.0  29459.0  29459.0           29378.0   
mean        3244.7   2019.4      6.4   1007.2              36.9   
std         2076.2      3.9      3.4   1832.3              17.7   
min           62.0   2013.0      1.0      4.0               0.0   
25%         1202.0   2016.0      3.0    235.0              24.0   
50%         3298.0   2019.0      6.0    445.0              35.0   
75%         5236.0   2023.0      9.0    859.0              48.0   
max         6831.0   2026.0     12.0  19553.0             168.0   

indikator  logiernaechte   zimmer  zimmerauslastung  zimmernaechte  
count            29378.0  29459.0           29378.0        29378.0  
mean             